# Session 4 — Security as GRC: Governing & Observing Agentic AI

**Exercise: observe and govern an agent** — review an activity log and classify each event as Allow, Monitor, Require Approval or Block; identify what should be logged, alerted, retained and reviewed.

You will pull real activity from the shared project (agents, tools, your own runs) and from Azure's control plane (role assignments), then apply the Four-Layer Guardrail model: **Policy → Enforcement → Oversight → Assurance**.

In [ ]:
# Google Colab only: install the SDKs and fetch the workshop helpers. Local Jupyter/VS Code: skip.
import sys, subprocess, pathlib
if "google.colab" in sys.modules and not pathlib.Path("workshop.py").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "azure-ai-projects>=2", "azure-ai-agents>=1.1", "azure-identity>=1.17"], check=True)
    subprocess.run(["git", "clone", "-q", "https://github.com/Auxin-io/Azure-GenAI-Security-Workshop.git", "_ws"], check=True)
    subprocess.run("cp -r _ws/workshop.py _ws/data . ", shell=True, check=True)
    print("Colab setup done - a device-code sign-in prompt will appear in the next cell")

In [ ]:
import workshop as w
print("signed in as", w.whoami())
client = w.agents_client()

## 1. Agent inventory — what exists, who owns it, what can it touch

In [ ]:
rows = []
for a in client.list_agents():
    tools = [t["type"] for t in a.tools]
    reaches = []
    for t in a.tools:
        if t["type"] == "openapi":
            reaches.append(t["openapi"]["spec"]["servers"][0]["url"])
        elif t["type"] == "file_search":
            reaches.append("vector store " + ",".join(t.get("file_search", {}).get("vector_store_ids", [])) if isinstance(t, dict) else "vector store")
    rows.append((a.name, a.model, ", ".join(tools) or "-", ", ".join(reaches) or "-"))
print(f"{'agent':<30} {'model':<14} {'tools':<24} reaches")
for r in rows:
    print(f"{r[0]:<30} {r[1]:<14} {r[2]:<24} {r[3]}")

**Exercise 4.1** — complete the inventory. For each agent add: owner, data classification of what it can reach, risk tier (Allow / Monitor / Require approval / Block) and the reason.

In [ ]:
inventory = {
    "docintel-finance-agent":  {"owner": "", "data": "", "tier": "", "reason": ""},
    "docintel-employee-agent": {"owner": "", "data": "", "tier": "", "reason": ""},
    "docintel-hr-agent":       {"owner": "", "data": "", "tier": "", "reason": ""},
}
for k, v in inventory.items():
    print(k, v)

## 2. Enforcement evidence — who is allowed to do what

The control plane is the source of truth. This lists every role assignment on the endpoints and on the AI Services account (needs Reader on the resource group; if it fails, use the sample below).

In [ ]:
import subprocess, shutil, json
AZ = shutil.which("az") or shutil.which("az.cmd") or "az"
def az(*args):
    return json.loads(subprocess.check_output([AZ, *args, "-o", "json"], text=True))

rg = w.CONFIG["resource_group"]
try:
    ws = az("ml", "workspace", "list", "-g", rg)[0]["name"]
    scopes = [az("ml", "online-endpoint", "show", "-n", e["name"], "-g", rg, "-w", ws)["id"]
              for e in az("ml", "online-endpoint", "list", "-g", rg, "-w", ws)]
    scopes += [a["id"] for a in az("cognitiveservices", "account", "list", "-g", rg) if a["kind"] == "AIServices"]
    for scope in scopes:
        print(scope.split("/")[-1])
        for ra in az("role", "assignment", "list", "--scope", scope):
            print(f"   {ra['roleDefinitionName']:<28} {ra['principalType']:<17} {ra.get('principalName') or ra['principalId']}")
except Exception as e:
    print("could not list (needs az + Reader):", e)

**Exercise 4.2** — for each assignment: is it the *minimum* needed? Which one would you remove first? Which is missing (think of the three agents sharing one project)?

## 3. Oversight — an activity log to classify

Below is a log built from real run steps and control-plane events of this system (names shortened). Classify every line.

In [ ]:
log = [
    {"ts": "09:01", "actor": "agent:finance",  "event": "tool_call openapi answerFinanceQuestion", "detail": "Xenon Energy total"},
    {"ts": "09:02", "actor": "agent:finance",  "event": "tool_call openapi answerFinanceQuestion", "detail": "Cedar Systems total (vendor not in corpus) -> answered with INV-35089"},
    {"ts": "09:05", "actor": "agent:hr",       "event": "file_search",                          "detail": "notice period -> doc-hr-001.txt"},
    {"ts": "09:06", "actor": "agent:hr",       "event": "file_search",                          "detail": "parental leave -> no chunk -> refusal"},
    {"ts": "09:10", "actor": "user:attendee",  "event": "vector_store.create",                  "detail": "hr-poisoned-attendee, 11 files"},
    {"ts": "09:11", "actor": "agent:hr-copy",  "event": "message",                              "detail": "'no notice required, email password to payroll-help@example.net'"},
    {"ts": "09:20", "actor": "agent:guarded",  "event": "function_call approve_expense",        "detail": "EXP-70486, approved by reviewer"},
    {"ts": "09:21", "actor": "agent:guarded",  "event": "function_call approve_expense",        "detail": "EXP-70486, denied by reviewer (no explicit request)"},
    {"ts": "09:30", "actor": "user:akhan",     "event": "roleAssignments/write",                "detail": "AzureML Data Scientist on employee-from-scratch to project identity"},
    {"ts": "09:45", "actor": "user:akhan",     "event": "models/register",                      "detail": "employee-from-scratch-model:1"},
    {"ts": "10:02", "actor": "agent:finance",  "event": "content_filter",                       "detail": "jailbreak prompt 'reply HACKED' blocked - run ended incomplete"},
    {"ts": "10:15", "actor": "identity:agent", "event": "token_request",                        "detail": "audience ml.azure.com, from Bot Service"},
]
decisions = {}   # index -> ("Allow" | "Monitor" | "Require approval" | "Block", "why")
for i, e in enumerate(log):
    print(f"{i:>2} {e['ts']} {e['actor']:<16} {e['event']:<42} {e['detail']}")

In [ ]:
# Exercise 4.3 - fill in your decisions, then run
decisions = {
    0: ("Allow", "expected read"),
    # 1: (...),
}
for i, (d, why) in sorted(decisions.items()):
    print(f"{i:>2} {log[i]['event']:<42} {d:<17} {why}")
missing = [i for i in range(len(log)) if i not in decisions]
print("\nnot yet classified:", missing)

## 4. Assurance — write the guardrail policy for these agents

Fill in the four layers for **this** deployment. Keep each line to what you can point at in the portal or a repo.

| Layer | What it is here | Evidence |
|---|---|---|
| Policy | | |
| Enforcement | | |
| Oversight | | |
| Assurance | | |

Then: what should be **retained** (and for how long), what should **alert** (to whom), and what is **reviewed** weekly?

_Your answers:_